In [1]:
from __future__ import annotations
from components import *
from checkpoint import *

import json
from pathlib import Path

from vllm_utils import *
from hf_utils import *

from drgrpo_grader import r1_zero_reward_fn
from torch.utils.data import DataLoader

from tqdm import tqdm

/home/sqsang/ml/assignment5-alignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configurations

In [2]:
model_id = "Qwen/Qwen2.5-0.5B"
# model_id = "EleutherAI/pythia-410m-deduped"

n_train_examples = 6400
n_val_examples = 1024
questions_per_rollout = 32

rollout_microbatch_size = 64
train_microbatch_size = 16
group_size = 16
sampling_max_tokens = 128

max_grad_norm = 1.0
learning_rate = 1e-5


sp = dict(
    temperature=1.0,
    n=group_size,
    max_tokens=sampling_max_tokens,
    top_p=1.0,
    seed=0,
    return_token_ids=True,
    stop=["</answer>"],
    include_stop_str_in_output=True
)


n_epochs = 1
device = "cuda:0"
eval_every = 10

gradient_accumulation_steps = questions_per_rollout * group_size // train_microbatch_size

## Data loader

In [3]:
def load_jsonl(path):
    with Path(path).open() as f:
        return [json.loads(line) for line in f]

data_path = Path("../data/gsm8k")
train_data = load_jsonl(data_path / "train.jsonl")[:n_train_examples]
test_data = load_jsonl(data_path / "test.jsonl")[:n_val_examples]

def repeat_list(lst, n):
    repeated_lst = []
    for item in lst:
        repeated_lst.extend([item] * n)
    return repeated_lst

def r1zero_prompt(question):
    prompt = f"A conversation between User and Assistant. The User asks a question, and the Assistant solvesit. The Assistant first thinks about the reasoning process in the mind and then provides theUser with the answer. The reasoning process is enclosed within <think> </think> and theanswer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoningprocess here </think> <answer> answer here </answer>.User: {question} Assistant: <think>"
    return prompt

def collate_gsm8k(examples):
    questions = [ex["question"] for ex in examples]
    answers = [ex["answer"].rstrip("\n").split("\n")[-1][5:] for ex in examples]
    prompts = [r1zero_prompt(q) for q in questions]
    return {
        "questions": questions,
        "prompts": prompts,
        "repeated_prompts": repeat_list(prompts, group_size),
        "answers": answers,
        "repeated_answers": repeat_list(answers, group_size),
    }

train_loader = DataLoader(
    train_data,
    batch_size=questions_per_rollout,
    shuffle=True,
    collate_fn=collate_gsm8k,
)

# test_loader = DataLoader(
#     test_data,
#     batch_size=len(test_data),
#     shuffle=False,
#     collate_fn=collate_gsm8k,
# )


def evaluate_llm(llm, reward_fn, prompt_fn, problems, eval_sampling_params):
    sp = eval_sampling_params.copy()
    sp["n"] = 1
    prompts = [prompt_fn(problem["question"]) for problem in problems]

    outputs = llm.generate_completions(prompts, sp)

    for i, problem in enumerate(problems):
        generated_text = outputs[i].text
        answer = problem["answer"].rstrip("\n").split("\n")[-1][5:]

        problem["generated_response"] = generated_text

        tmp = reward_fn(generated_text, answer)
        problem["format_reward"] = tmp["format_reward"]
        problem["reward"] = tmp["reward"]
    
    avg_reward = sum([problem["reward"] for problem in problems]) / len(problems)
    pad_token_id = llm.tokenizer.pad_token_id
    avg_response_length = sum(
            sum(token_id != pad_token_id for token_id in output.token_ids)
            for output in outputs
        ) / len(outputs)

    print(f"Average reward: {avg_reward:.4f}")
    print(f"Average response length: {avg_response_length:.2f} tokens")
    return avg_reward, avg_response_length

## Load model

In [4]:
model, tokenizer = get_model_and_tokenizer(model_id, device=device)
generator = HFCompletionServer(model, tokenizer)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.0)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 290/290 [00:00<00:00, 3365.35it/s]


## Training loop

In [5]:
loss_list = []
reward_list = []
eval_reward_list = []
eval_response_length_list = []

# for i, batch in tqdm(enumerate(train_loader)):
for i, batch in enumerate(train_loader):
    prompts = batch["prompts"]

    print(f"[{i}/{len(train_loader)}][Sampling...]", end="")
    rollout_responses = generator.generate_completions(prompts, sp, rollout_microbatch_size)
    rollout_responses = [resp.text for resp in rollout_responses]


    print("[Training...]", end="\t" )
    model.train()
    tot_loss, stats = grpo_train_step(
        model=model,
        tokenizer=tokenizer,
        optimizer=optimizer,
        gradient_accumulation_steps=gradient_accumulation_steps,
        max_grad_norm=max_grad_norm,
        reward_fn=r1_zero_reward_fn,
        repeated_prompts=batch["repeated_prompts"],
        rollout_responses=rollout_responses,
        repeated_ground_truths=batch["repeated_answers"],
        group_size=group_size,
    )
    
    print(f"Loss: {tot_loss:.4f}, Avg Rewards: {stats['avg_rewards']:.4f}")
    loss_list.append(tot_loss)
    reward_list.append(stats["avg_rewards"])

    if (i +1) % eval_every == 0:
        print("="*50)
        print("Evaluating...")
        model.eval()
        avg_reward, avg_response_length = evaluate_llm(
            llm=generator,
            reward_fn=r1_zero_reward_fn,
            prompt_fn=r1zero_prompt,
            problems=test_data,
            eval_sampling_params=sp,
        )
        eval_reward_list.append(avg_reward)
        eval_response_length_list.append(avg_response_length)
        print("="*50)



[0/200][Sampling...][Training...]	Loss: -0.0086, Avg Rewards: 0.0078
[1/200][Sampling...][Training...]	Loss: 0.0000, Avg Rewards: 0.0000
[2/200][Sampling...][Training...]	Loss: -0.0020, Avg Rewards: 0.0020
[3/200][Sampling...][Training...]	Loss: 0.0000, Avg Rewards: 0.0000
[4/200][Sampling...][Training...]	Loss: -0.0041, Avg Rewards: 0.0020
[5/200][Sampling...][Training...]	Loss: 0.0000, Avg Rewards: 0.0000
[6/200][Sampling...][Training...]	Loss: 0.0000, Avg Rewards: 0.0000
[7/200][Sampling...][Training...]	Loss: -0.0011, Avg Rewards: 0.0020
[8/200][Sampling...][Training...]	Loss: 0.0017, Avg Rewards: 0.0020
[9/200][Sampling...][Training...]	Loss: 0.0000, Avg Rewards: 0.0000
Evaluating...
Average reward: 0.0020
Average response length: 111.11 tokens
[10/200][Sampling...][Training...]	Loss: -0.0062, Avg Rewards: 0.0039
[11/200][Sampling...][Training...]	Loss: -0.0038, Avg Rewards: 0.0039
[12/200][Sampling...][Training...]	Loss: -0.0069, Avg Rewards: 0.0059
[13/200][Sampling...][Training